# Module 1: Building a Simple Corporate Travel Agent

## What We Are Building

Contoso Corp needs a travel assistant that helps employees book business trips.
Employees ask things like:
- "Find me flights to New York next week"
- "What hotels are available there?"
- "What's our travel policy for my level?"

We will build this agent using the **Microsoft Agent Framework** — a production-grade
framework for creating AI agents that can reason, use tools, and hold conversations.

## Why This Notebook Exists

Before we can discuss memory, we need a working agent to observe.
This notebook builds one, has a real conversation with it, and then exposes
the exact moment where lack of persistent memory causes failure.

By the end you will have seen:
1. A working agent with tools (search flights, hotels, policies)
2. Chat history working perfectly within a conversation
3. Chat history failing completely across conversations
4. Why this failure matters in production


## Prerequisites

- Python 3.11+ with the shared `.venv` activated
- `az login` completed (Azure CLI credential)
- A `.env` file at the repo root with `FOUNDRY_PROJECT_ENDPOINT` set
- Microsoft Agent Framework installed via `requirements.txt`


In [ ]:
%pip install -q -r ../requirements.txt


## Step 1: Connect to Azure AI Foundry

Every agent needs a language model to reason with. The Microsoft Agent Framework
connects to Azure AI Foundry via `FoundryChatClient`. This client handles:
- Authentication (we use Azure CLI credentials)
- Model selection (which deployment to use)
- Request routing to your Foundry project endpoint

Think of it as plugging the agent's brain in.


In [ ]:
import os
import json
import sniffio
from pathlib import Path
from dotenv import load_dotenv

sniffio.current_async_library_cvar.set("asyncio")
load_dotenv("../.env", override=True)

PROJECT_ENDPOINT = os.environ["FOUNDRY_PROJECT_ENDPOINT"]
MODEL = os.environ.get("FOUNDRY_MODEL", "gpt-4o")

print(f"Endpoint: {PROJECT_ENDPOINT}")
print(f"Model: {MODEL}")

In [ ]:
from azure.identity import AzureCliCredential
from agent_framework import Agent, AgentSession, tool
from agent_framework.foundry import FoundryChatClient

credential = AzureCliCredential()
client = FoundryChatClient(
    project_endpoint=PROJECT_ENDPOINT,
    model=MODEL,
    credential=credential,
)
print("Client ready")


## Step 2: Give the Agent Tools

An agent that can only chat is limited. A travel assistant needs to **act**:
search for flights, look up hotels, check company policies.

In the Microsoft Agent Framework, tools are regular Python functions decorated
with `@tool`. The framework:
1. Tells the model what tools are available (via function descriptions)
2. The model decides when a tool is needed based on the conversation
3. The framework calls your function and feeds the result back to the model

This is how agents go from "I can discuss travel" to "I can find you a flight."


In [ ]:
# Load the sample data our tools will query
data_dir = Path("../data")
flights = json.loads((data_dir / "flights.json").read_text(encoding="utf-8"))
hotels = json.loads((data_dir / "hotels.json").read_text(encoding="utf-8"))
policies = json.loads((data_dir / "travel_policies.json").read_text(encoding="utf-8"))

print(f"Loaded: {len(flights)} flights, {len(hotels)} hotels")


In [ ]:
CITY_TO_AIRPORT = {
    "new york": "JFK", "nyc": "JFK", "manhattan": "JFK",
    "london": "LHR", "heathrow": "LHR",
    "san francisco": "SFO", "sf": "SFO",
    "tokyo": "NRT", "narita": "NRT",
    "seattle": "SEA",
}

@tool
async def search_flights(destination: str) -> str:
    """Search available flights to a destination city."""
    code = CITY_TO_AIRPORT.get(destination.lower(), destination.upper())
    results = [f for f in flights if code == f["destination"].upper()]
    if not results:
        return f"No flights found to {destination}"
    return json.dumps(results[:3], indent=2)

In [ ]:
@tool
async def search_hotels(city: str) -> str:
    """Search available hotels in a city."""
    results = [h for h in hotels if city.lower() in h["city"].lower()]
    if not results:
        return f"No hotels found in {city}"
    return json.dumps(results[:3], indent=2)


In [ ]:
@tool
async def get_travel_policy(employee_level: str) -> str:
    """Get travel policy rules for an employee level (e.g. IC5, M1, Director)."""
    level_policies = policies.get("by_level", {}).get(employee_level, {})
    if not level_policies:
        return f"No policy found for level: {employee_level}"
    return json.dumps(level_policies, indent=2)


## Step 3: Create the Travel Agent

Now we assemble the pieces:
- **Client** — the language model connection
- **System prompt** — defines who the agent is and how it behaves
- **Tools** — the actions it can take

The result is a fully functional travel assistant that can reason about
requests and use tools to fulfill them.


In [ ]:
SYSTEM_PROMPT = """You are a corporate travel assistant for Contoso Corp.
Help employees book business travel including flights and hotels.
Use the available tools to search for options whenever a destination is mentioned.
Do not ask clarifying questions — just search with the information provided and present results.
Be concise and helpful."""

travel_agent = Agent(
    client=client,
    name="TravelAssistant",
    instructions=SYSTEM_PROMPT,
    tools=[search_flights, search_hotels, get_travel_policy],
)
print("Travel agent created with 3 tools")

## Step 4: Have a Conversation (Chat History Works)

The Microsoft Agent Framework keeps **chat history** via `AgentSession`.
When you pass the same session object to each `agent.run()` call, the framework
keeps the full conversation history and sends it to the model every turn.

This means the agent can:
- Remember your name from turn 1 in turn 4
- Resolve "there" to a city mentioned earlier
- Build on prior tool results

This is **chat history** — built into the framework, automatic, and effective.
But it is not memory. Let’s watch it work, then watch it break.


In [ ]:
async def booking_conversation():
    session = AgentSession()
    turns = [
        "Hi, I'm Sarah Chen from Engineering. I need to travel to New York next week for a conference.",
        "Can you find me flights there?",
        "What about hotels in that city?",
        "What's my name, where am I going, and why?",
    ]
    for message in turns:
        print(f"User: {message}")
        r = await travel_agent.run(message, session=session)
        print(f"Agent: {r.text}\n")

await booking_conversation()

### What Just Happened

The agent tracked everything within that conversation:
- **Your name** (Sarah Chen) — stated in turn 1, recalled in turn 4
- **Your destination** (New York) — stated in turn 1, resolved from "there" in turn 2
- **Context continuity** — "What about hotels?" knew to search New York

This works because `AgentSession` accumulates the full message history.
Every `agent.run()` call sends all prior turns to the model, so it always
has complete context.

Chat history is powerful within a session. But it has a critical boundary.


## Step 5: Watch Chat History Break

Chat history lives inside the `AgentSession` Python object.
It is **transient** — it exists only as long as that object exists.

When any of these happen, chat history is gone:
- The user closes the browser and returns tomorrow
- The application restarts or redeploys
- The kernel restarts (try it after this cell!)
- A different agent instance handles the next request

This is not a bug — it is by design. The framework gives you fast,
in-process conversation context. It does not persist anything to disk
or database unless you add that yourself.

Let’s prove it: we create a **new session** and ask the same questions.


In [ ]:
async def new_session_test():
    # Brand new session — no history from the booking conversation
    fresh_session = AgentSession()

    # Ask something only knowable from the previous session
    print("User: Can you remind me what city my conference is in next week?")
    r = await travel_agent.run(
        "Can you remind me what city my conference is in next week?",
        session=fresh_session,
    )
    print(f"Agent: {r.text}\n")

    # Ask for identity — the agent has no idea
    print("User: You helped me book this trip earlier. What's my name?")
    r = await travel_agent.run(
        "You helped me book this trip earlier. What's my name?",
        session=fresh_session,
    )
    print(f"Agent: {r.text}")

await new_session_test()

### The Problem Is Now Visible

The agent has no idea who you are. It cannot recall your name, your
destination, or your preferences. From its perspective, you are a
complete stranger.

This is the **production reality**:
- Users expect continuity across sessions ("Book my usual trip")
- The system provides amnesia across sessions
- Every new session starts from zero

**Chat history is necessary but not sufficient.**
It handles within-conversation continuity perfectly.
It provides zero across-conversation continuity.


## Summary

### What Works vs What Breaks

| What Works | What Breaks |
|---|---|
| Multi-turn context within a session | History lost when session ends |
| Tool calling (flights, hotels, policy) | No recall of past preferences |
| Resolving references ("there", "my name") | Cannot learn from prior sessions |

### Architecture So Far

```mermaid
flowchart LR
  U[User] --> A[Agent + Tools]
  A --> H[Chat History]
  H -->|within session| A
  H -->|session ends| X[Lost forever]
```

### Next Step

Module 2 introduces **persistent chat history** backed by Cosmos DB and
**context engineering** techniques like summarization. We’ll see how far
we can get — and why even persistent, compacted history is still not memory.
